In [1]:
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import uniform_filter1d
import os
import matplotlib.pyplot as plt

In [2]:
### GLOBAL PATH INPUTS

pData = 'data/'
pFCC  = pData + 'FCC-21X21/'
pTri  = pData + 'Tri-21X21/'
pHex  = pData + 'Hex-21X21/'
pKag  = pData + 'Kag-21X21/'

pAK          = pData + 'AK/'
pUTdisNodes  = pAK + 'Ductile-disNodes-FCC-12X16/'
pUTdisStruts = pAK + 'Ductile-disStruts-FCC-12X16/'
pFTdisNodes  = pAK + 'Fracture-disNodes/'

# MATLAB to CSV Data File Conversion

In [3]:
### INPUT

dir1 = pUTdisNodes
mode = 'Ductile'
dis = 'disNodes'

MATin  = dir1 + 'raw/' + 'inputData-20-frame-0.mat'
MATout = dir1 + 'raw/' + 'outputStressStrain-20-1-1500.mat'

CSVin  = dir1 + f'{mode}-{dis}-IN.csv'
CSVout = dir1 + f'{mode}-{dis}-OUT.csv'
append = False

run1 = False

In [4]:
def mat_to_csv(MATin, MATout, CSVin, CSVout, append=False):
    input_mat = scipy.io.loadmat(MATin)
    in_data = pd.DataFrame(list(input_mat.values())[3])
    in_data = in_data.rename(columns={i:str(i) for i in range(len(in_data.columns))})
    if append:
        in_header = pd.read_csv(CSVin, index_col=0)
        in_header = in_header.rename(columns={i:str(i) for i in range(len(in_header.columns))})
        in_data = in_data + in_header.iloc[0].values
        in_data = pd.concat([in_header, in_data], ignore_index=True)
    in_data.to_csv(CSVin)

    output_mat = scipy.io.loadmat(MATout)
    out_data = pd.DataFrame(list(output_mat.values())[3])
    out_data = out_data.rename(columns={i:str(i) for i in range(len(out_data.columns))})
    if append:
        out_header = pd.read_csv(CSVout, index_col=0)
        out_header = out_header.drop(out_header.columns[201:], axis=1)
        out_header = out_header.rename(columns={i:str(i) for i in range(len(out_header.columns))})
        out_data = pd.concat([out_header, out_data], ignore_index=True)
    out_data.to_csv(CSVout)

In [5]:
if run1:
    a = mat_to_csv(MATin, MATout, CSVin, CSVout, append=append)

# Create Input and Output CSV from Raw .txt Files

### Inputs

In [22]:
### INPUT

dir2 = pFCC

run2 = True

In [23]:
def get_nodes(inpFile):
    with open(inpFile, 'r') as f:
        lines = f.readlines()
        
    node_lines = [line.split(',') for line in lines]
    
    nodes = [[float(elem.strip('\n').strip('.').strip()) for elem in line] for line in node_lines]
    nodesCoords = [node[1:] for node in nodes]
    nodes_df = pd.DataFrame(nodesCoords, columns=['x','y'])
    
    return nodes, nodesCoords, nodes_df

def get_struts(inpFile):
    with open(inpFile, 'r') as f:
        lines = f.readlines()
    
    thicks = [float(line.strip('\n')) for line in lines]
    
    return thicks

def create_inputCSV(directory):
    duct_disNodes = []
    duct_disStruts = []
    frac_disNodes = []
    frac_disStruts = []
    
    for file in os.listdir(directory):
        pathRaw = directory + file
        if file == "per":
            for ffile in os.listdir(directory + file):
                if (ffile.endswith('.txt') and 'Ductile' in ffile):
                    if 'IN-n' in ffile:
                        nodes, nodesCoords, nodes_df = get_nodes(pathRaw + '/' + ffile)
                        duct_disNodes.insert(0, nodes_df.to_numpy().flatten())
                    elif 'IN-s' in ffile:
                        thicks = get_struts(pathRaw + '/' + ffile)
                        duct_disStruts.insert(0, thicks)
                    else:
                        pass
                elif (ffile.endswith('.txt') and 'Fracture' in ffile):
                    if 'IN-n' in ffile:
                        nodes, nodesCoords, nodes_df = get_nodes(pathRaw + '/' + ffile)
                        frac_disNodes.insert(0, nodes_df.to_numpy().flatten())
                    elif 'IN-s' in ffile:
                        thicks = get_struts(pathRaw + '/' + ffile)
                        frac_disStruts.insert(0, thicks)
                    else:
                        pass
                else:
                    pass
        elif ("per" not in file and ".csv" not in file and ".ipynb" not in file):
            for ffile in os.listdir(directory + file):
                if (ffile.endswith('.txt') and 'Ductile' in ffile and 'IN-' in ffile):
                    if 'disNodes' in ffile:
                        nodes, nodesCoords, nodes_df = get_nodes(pathRaw + '/' + ffile)
                        duct_disNodes.append(nodes_df.to_numpy().flatten())
                    elif 'disStruts' in ffile:
                        thicks = get_struts(pathRaw + '/' + ffile)
                        duct_disStruts.append(thicks)
                    else:
                        pass
                elif (ffile.endswith('.txt') and 'Fracture' in ffile and 'IN-' in ffile):
                    if 'disNodes' in ffile:
                        nodes, nodesCoords, nodes_df = get_nodes(pathRaw + '/' + ffile)
                        frac_disNodes.append(nodes_df.to_numpy().flatten())
                    elif 'disStruts' in ffile:
                        thicks = get_struts(pathRaw + '/' + ffile)
                        frac_disStruts.append(thicks)
                    else:
                        pass
                else:
                    pass
        else:
            pass
    
    UTdisNodesIN_df = pd.DataFrame(duct_disNodes)
    UTdisStrutsIN_df = pd.DataFrame(duct_disStruts)
    FTdisNodesIN_df = pd.DataFrame(frac_disNodes)
    FTdisStrutsIN_df = pd.DataFrame(frac_disStruts)
    
    UTdisNodesIN_df.to_csv(directory + '/Ductile-disNodes-IN.csv')
    UTdisStrutsIN_df.to_csv(directory + '/Ductile-disStruts-IN.csv')
    FTdisNodesIN_df.to_csv(directory + '/Fracture-disNodes-IN.csv')
    FTdisStrutsIN_df.to_csv(directory + '/Fracture-disStruts-IN.csv')
    
    return UTdisNodesIN_df, UTdisStrutsIN_df, FTdisNodesIN_df, FTdisStrutsIN_df

In [24]:
if run2:
    UTdisNodesIN_df, UTdisStrutsIN_df, FTdisNodesIN_df, FTdisStrutsIN_df = create_inputCSV(dir2)

In [9]:
# import os

# def export_struts(inpFile, expFile, dim):
#     with open(inpFile, 'r') as f:
#         lines = f.readlines()

#     strut_lines = [lines[lines.index(line)+1].split(' ') for line in lines if "*Beam Section, elset=edgeElem-" in line]
#     thicks = [line[-1] for line in strut_lines]
    
#     with open(expFile, 'w') as f:
#         for thick in thicks:
#             f.write(thick)


# os.mkdir('data/transfer')
# for file in os.scandir('data'):
#     if file.name.endswith('.inp'):
#         expFile = "transfer/IN-" + file.name[:-4].replace('_','-') + ".txt"
#         export_struts('data/' + file.name, 'data/' + expFile, [21,21])
#     else:
#         pass

# get_struts('data/transfer/IN-Ductile-FCC-21X21-disStruts-1.txt')

### Outputs

In [10]:
### INPUT

dir3 = pFCC

run3 = False

In [11]:
def get_outputCurve(outputFile):
    with open(outputFile, 'r') as f:
        lines = f.readlines()
    
    lines = [line.split(' ') for line in lines]
    
    output = [[float(elem.strip('\n').strip('.').strip()) for elem in line] for line in lines]
    outputXY = [node[1:] for node in output]
    output_df = pd.DataFrame(outputXY, columns=['x','y'])
    
    return output, outputXY, output_df

def create_outputCSV(directory):
    duct_disNodes = []
    duct_disStruts = []
    frac_disNodes = []
    frac_disStruts = []
    
    for file in os.listdir(directory):
        pathRaw = directory + file
        if file == "per":
            for ffile in os.listdir(directory + file):
                if (ffile.endswith('.txt') and 'Ductile' in ffile and 'OUT-' in ffile):
                    output, outputXY, output_df = get_outputCurve(pathRaw + '/' + ffile)
                    duct_disNodes.insert(0, output_df.x.tolist())
                    duct_disStruts.insert(0, output_df.x.tolist())
                    duct_disNodes.insert(1, uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                    duct_disStruts.insert(1, uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                elif (ffile.endswith('.txt') and 'Fracture' in ffile and 'OUT-' in ffile):
                    output, outputXY, output_df = get_outputCurve(pathRaw + '/' + ffile)
                    frac_disNodes.insert(0, output_df.x.tolist())
                    frac_disStruts.insert(0, output_df.x.tolist())
                    frac_disNodes.insert(1, uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                    frac_disStruts.insert(1, uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                else:
                    pass
        elif ("per" not in file and ".csv" not in file and ".ipynb" not in file):
            for ffile in os.listdir(directory + file):
                if (ffile.endswith('.txt') and 'Ductile' in ffile and 'OUT-' in ffile):
                    if 'disNodes' in ffile:
                        output, outputXY, output_df = get_outputCurve(pathRaw + '/' + ffile)
                        duct_disNodes.append(uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                    elif 'disStruts' in ffile:
                        output, outputXY, output_df = get_outputCurve(pathRaw + '/' + ffile)
                        duct_disStruts.append(uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                    else:
                        pass
                elif (ffile.endswith('.txt') and 'Fracture' in ffile and 'OUT-' in ffile):
                    if 'disNodes' in ffile:
                        output, outputXY, output_df = get_outputCurve(pathRaw + '/' + ffile)
                        frac_disNodes.append(uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                    elif 'disStruts' in ffile:
                        output, outputXY, output_df = get_outputCurve(pathRaw + '/' + ffile)
                        frac_disStruts.append(uniform_filter1d(output_df.y.tolist(), size=10, mode='nearest'))
                    else:
                        pass
                else:
                    pass
        else:
            pass
    
    UTdisNodesOUT_df = pd.DataFrame(duct_disNodes)
    UTdisStrutsOUT_df = pd.DataFrame(duct_disStruts)
    FTdisNodesOUT_df = pd.DataFrame(frac_disNodes)
    FTdisStrutsOUT_df = pd.DataFrame(frac_disStruts)
    
    UTdisNodes_drop = [i for i,j in enumerate(UTdisNodesOUT_df.isnull().any(axis=1)) if j == True]
    UTdisNodesOUT_df = UTdisNodesOUT_df.drop(UTdisNodes_drop, axis=0).reset_index(drop=True)
    UTdisNodes_drop = [i-1 for i in UTdisNodes_drop]
    UTdisNodesIN_df = pd.read_csv(directory + 'Ductile-disNodes-IN.csv', index_col=0)
    UTdisNodesIN_df = UTdisNodesIN_df.drop(UTdisNodes_drop, axis=0).reset_index(drop=True)
    UTdisNodesOUT_df.to_csv(directory + 'Ductile-disNodes-OUT.csv')
    UTdisNodesIN_df.to_csv(directory + 'Ductile-disNodes-IN.csv')
    
    UTdisStruts_drop = [i for i,j in enumerate(UTdisStrutsOUT_df.isnull().any(axis=1)) if j == True]
    UTdisStrutsOUT_df = UTdisStrutsOUT_df.drop(UTdisStruts_drop, axis=0).reset_index(drop=True)
    UTdisStruts_drop = [i-1 for i in UTdisStruts_drop]
    UTdisStrutsIN_df = pd.read_csv(directory + 'Ductile-disStruts-IN.csv', index_col=0)
    UTdisStrutsIN_df = UTdisStrutsIN_df.drop(UTdisStruts_drop, axis=0).reset_index(drop=True)
    UTdisStrutsOUT_df.to_csv(directory + 'Ductile-disStruts-OUT.csv')
    UTdisStrutsIN_df.to_csv(directory + 'Ductile-disStruts-IN.csv')
    
    FTdisNodes_drop = [i for i,j in enumerate(FTdisNodesOUT_df.isnull().any(axis=1)) if j == True]
    FTdisNodesOUT_df = FTdisNodesOUT_df.drop(FTdisNodes_drop, axis=0).reset_index(drop=True)
    FTdisNodes_drop = [i-1 for i in FTdisNodes_drop]
    FTdisNodesIN_df = pd.read_csv(directory + 'Fracture-disNodes-IN.csv', index_col=0)
    FTdisNodesIN_df = FTdisNodesIN_df.drop(FTdisNodes_drop, axis=0).reset_index(drop=True)
    FTdisNodesOUT_df.to_csv(directory + 'Fracture-disNodes-OUT.csv')
    FTdisNodesIN_df.to_csv(directory + 'Fracture-disNodes-IN.csv')
    
    FTdisStruts_drop = [i for i,j in enumerate(FTdisStrutsOUT_df.isnull().any(axis=1)) if j == True]
    FTdisStrutsOUT_df = FTdisStrutsOUT_df.drop(FTdisStruts_drop, axis=0).reset_index(drop=True)
    FTdisStruts_drop = [i-1 for i in FTdisStruts_drop]
    FTdisStrutsIN_df = pd.read_csv(directory + 'Fracture-disStruts-IN.csv', index_col=0)
    FTdisStrutsIN_df = FTdisStrutsIN_df.drop(FTdisStruts_drop, axis=0).reset_index(drop=True)
    FTdisStrutsOUT_df.to_csv(directory + 'Fracture-disStruts-OUT.csv')
    FTdisStrutsIN_df.to_csv(directory + 'Fracture-disStruts-IN.csv')
    
    return UTdisNodesOUT_df, UTdisStrutsOUT_df, FTdisNodesOUT_df, FTdisStrutsOUT_df

def update_inputsCSV(directory):
    pass

In [12]:
if run3:
    UTdisNodesOUT_df, UTdisStrutsOUT_df, FTdisNodesOUT_df, FTdisStrutsOUT_df = create_outputCSV(dir3)

# Load Existing CSV Data

In [13]:
### INPUTS

dir4 = pFCC
mode = 'Ductile'      # Ductile, Fracture
dis  = 'disNodes'     # disNodes, disStruts

CSVin = dir4 + f'{mode}-{dis}-IN.csv'
CSVout = dir4 + f'{mode}-{dis}-OUT.csv'
typ = 'n'

run4 = True

In [14]:
def load_data(inputs, outputs, typ='n'):
    if typ == 'a':
        dIN_df = pd.read_csv(inputs, index_col=0)
        dOUT_df = pd.read_csv(outputs, index_col=0)
        
        IN_df, OUT_df, perIN_df, perOUT_df = None, None, None, None
        
    elif typ == 'n':
        IN_df = pd.read_csv(inputs, index_col=0)
        OUT_df = pd.read_csv(outputs, index_col=0)
        
        perIN_df = IN_df.iloc[:1].T
        perIN_df = perIN_df.rename(columns={0: "in"})
        perOUT_df = OUT_df.iloc[:2].T
        perOUT_df = perOUT_df.rename(columns={0: "x", 1: "y"})
        
        dIN_df = IN_df - IN_df.iloc[0].values
        IN_df = IN_df.reset_index(drop=True)
        dIN_df = dIN_df.reset_index(drop=True)
        dIN_df = dIN_df.loc[:, ~(dIN_df == 0.0).all()]
        dOUT_df = OUT_df - OUT_df.iloc[1].values
        OUT_df = OUT_df.iloc[1:].reset_index(drop=True)
        dOUT_df = dOUT_df.iloc[1:].reset_index(drop=True)
        
    else:
        raise ValueError("Incorrect data type, please specify from the following: a, n.")
    
    return IN_df, OUT_df, perIN_df, perOUT_df, dIN_df, dOUT_df

In [15]:
if run4:
    IN_df, OUT_df, perIN_df, perOUT_df, dIN_df, dOUT_df = load_data(CSVin, CSVout, typ=typ)

In [16]:
# plt.plot(perOUT_df.x.to_list(), OUT_df.iloc[0].tolist(), label='t')
# plt.legend()

In [17]:
# def get_boundaries(nodes_df):
#     x0 = min(nodes_df.x.tolist())
#     x1 = max(nodes_df.x.tolist())
#     y0 = min(nodes_df.y.tolist())
#     y1 = max(nodes_df.y.tolist())
    
#     left = nodes_df[nodes_df.x == x0]
#     right = nodes_df[nodes_df.x == x1]
#     bot = nodes_df[nodes_df.y == y0]
#     top = nodes_df[nodes_df.y == y1]
    
#     boundary_df = left.append(right).append(bot).append(top)
    
#     return boundary_df

# def get_nonBoundaries(nodes_df, boundary_df):
#     nonBoundary_df = nodes_df.drop(boundary_df.index)
    
#     return nonBoundary_df

# perNodes, perNodesCoords, perNodes_df = get_nodes(per_inpFile)
# disNodes, disNodesCoords, disNodes_df = get_nodes(dis_inpFile)

# perBoundary_df = get_boundaries(perNodes_df)
# disBoundary_df = get_boundaries(disNodes_df)

# perNonBoundary_df = get_nonBoundaries(perNodes_df, perBoundary_df)
# disNonBoundary_df = get_nonBoundaries(disNodes_df, disBoundary_df)

Normalize and then delta ??

Any struts unvaried ??